# Qwen3.5-0.8B Fine-tuning — LoRA + wandb

**What this does:**
- Load Qwen3.5-0.8B-Base (or Instruct) with 4-bit quantization (QLoRA)
- Fine-tune on your personal dataset (Starfire conversations or any text)
- Log metrics to wandb
- Save merged adapter weights

**Colab secrets needed:**
- `HF_TOKEN` — HuggingFace auth token
- `WANDB_API_KEY` — Weights & Biases (optional, for logging)
- `GH_TOKEN` — GitHub token (if loading private datasets)

**Runtime:** T4 (free tier) — ~15-20GB VRAM with QLoRA

In [ ]:
# ══════════════════════════════════════════════════════════════
# SETUP — install deps, auth, wandb init
# ══════════════════════════════════════════════════════════════

!pip install transformers accelerate bitsandbytes peft trl huggingface_hub wandb -q

import os
from google.colab import userdata

# ── HuggingFace token ──────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
if HF_TOKEN:
    print(f"HF_TOKEN loaded: {HF_TOKEN[:8]}...")
else:
    raise RuntimeError("HF_TOKEN not found. Add to Colab Secrets.")

# ── GitHub token (if needed for private datasets) ─────────────
GH_TOKEN = os.environ.get('GH_TOKEN') or userdata.get('GH_TOKEN')
if GH_TOKEN:
    os.environ['GH_TOKEN'] = GH_TOKEN
    print(f"GH_TOKEN loaded: {GH_TOKEN[:8]}...")
else:
    print("GH_TOKEN not found (optional — for private datasets)")

# ── wandb init ───────────────────────────────────────────────
WANDB_KEY = os.environ.get('WANDB_API_KEY') or userdata.get('WANDB_API_KEY')
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    wandb.init(project="qwen-finetune", name="qwen-0.8b-starfire")
    print("wandb initialized")
else:
    print("WANDB_API_KEY not found — logging disabled")

# Login to HF (needed for gated models)
from huggingface_hub import login
login(token=HF_TOKEN)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## ══════════════════════════════════════════════════════════════
## CONFIG

In [ ]:
from dataclasses import dataclass, field

@dataclass
class FinetuneConfig:
    # Model
    model_name: str = "Qwen/Qwen3.5-0.8B-Base"
    # model_name: str = "Qwen/Qwen3.5-0.8B"  # Use this for Instruct variant
    
    # Quantization (QLoRA)
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"        # nf4 or fp4
    bnb_4bit_compute_dtype: str = "bfloat16"
    bnb_4bit_use_double_quant: bool = True
    
    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])
    
    # Training
    batch_size: int = 2
    grad_accum_steps: int = 8
    epochs: int = 3
    learning_rate: float = 2e-4
    warmup_steps: int = 100
    weight_decay: float = 0.01
    max_seq_length: int = 512
    
    # Logging
    logging_steps: int = 10
    save_steps: int = 200
    eval_steps: int = 200
    
    # Data
    dataset_path: str = "ZacharyMaronek/starfire-personal-v1"  # or local path
    dataset_format: str = "starfire"  # "starfire" | "alpaca" | "chatml"

    # Output
    output_dir: str = "./qwen-finetuned"
    run_name: str = "qwen-0.8b-starfire-v1"

config = FinetuneConfig()
print(f"Model: {config.model_name}")
print(f"Load in 4bit: {config.load_in_4bit}")
print(f"Max sequence length: {config.max_seq_length}")
print(f"Effective batch size: {config.batch_size * config.grad_accum_steps}")

## ══════════════════════════════════════════════════════════════
## DATASET LOADING

Supports three formats:
- `starfire` — Zachary's Starfire conversation JSONL
- `alpaca` — standard Alpaca format {"instruction", "input", "output"}
- `chatml` — ChatML format {"messages": [{"role", "content"}]}

If using a private GH repo: `GH_TOKEN` env var enables private access.

In [ ]:
from datasets import load_dataset
import re

def load_starfires_format(dataset_path: str):
    """
    Load Zachary's Starfire personal training data.
    Supports:
    - HuggingFace dataset: ZacharyMaronek/starfire-personal-v1
    - Local JSONL file: ./starfire_training_data.txt
    - GitHub private repo via GH_TOKEN
    """
    if dataset_path.startswith("http") or "/" in dataset_path:
        # HuggingFace dataset
        print(f"Loading from HuggingFace: {dataset_path}")
        ds = load_dataset(dataset_path, split="train")
    elif os.path.isfile(dataset_path):
        # Local file
        print(f"Loading from local file: {dataset_path}")
        ds = load_dataset("json", data_files=dataset_path, split="train")
    else:
        # Try as HF dataset path
        print(f"Loading from HuggingFace: {dataset_path}")
        ds = load_dataset(dataset_path, split="train")

    print(f"Loaded {len(ds)} examples")
    return ds

def format_starfire_conversation(example):
    """
    Convert Starfire conversation format to ChatML.
    Format in dataset: {"input": "user message", "output": "assistant response"}
    """
    messages = []
    
    if "instruction" in example and "output" in example:
        # Alpaca format
        if example.get("instruction"):
            messages.append({"role": "user", "content": example["instruction"]})
        if example.get("input"):
            messages[-1]["content"] += f"\n{example['input']}"
        if example.get("output"):
            messages.append({"role": "assistant", "content": example["output"]})
    elif "messages" in example:
        # ChatML format already
        return example["messages"]
    elif "input" in example and "output" in example:
        # Starfire format
        messages.append({"role": "user", "content": str(example["input"])})
        messages.append({"role": "assistant", "content": str(example["output"])})
    
    return {"messages": messages}

def extract_text_from_messages(example):
    """Extract plain text from messages for base model training."""
    if "messages" in example:
        text = ""
        for msg in example["messages"]:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            text += f"<|{role}|> {content} "
        text += "<|end|>

In [ ]:
# ─── Load and inspect dataset ─────────────────────────────────

dataset = load_starfires_format(config.dataset_path)
print(f"\nDataset columns: {dataset.column_names}")
print(f"\nFirst example:")
example = dataset[0]
for k, v in example.items():
    if isinstance(v, str) and len(v) > 200:
        print(f"  {k}: {v[:200]}...")
    else:
        print(f"  {k}: {v}")

# Format all examples
formatted = dataset.map(format_starfire_conversation)
print(f"\nFormatted {len(formatted)} examples")

# Filter out empty conversations
def is_valid(example):
    msgs = example.get("messages", [])
    return len(msgs) >= 2 and any(m.get("content") for m in msgs)

filtered = formatted.filter(is_valid)
print(f"Valid examples: {len(filtered)}")

# Split train/val
split = filtered.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds = split["test"]
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## ══════════════════════════════════════════════════════════════
## LOAD MODEL — QLoRA (4-bit quantization)

In [ ]:
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ─── Quantization config ──────────────────────────────────────
quantization_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
    bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
    use_fast=False,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model with QLoRA...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\nModel loaded. Memory: {model.get_memory_footprint()/1e9:.1f} GB")
print(f"Model dtype: {model.dtype}")

## ══════════════════════════════════════════════════════════════
## PREPARE MODEL FOR TRAINING — add LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# ─── LoRA config ──────────────────────────────────────────────
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=config.lora_target_modules,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("Applying LoRA adapters...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()
model.config.use_cache = False  # Disable KV cache during training

print(f"\nTrainable params: {model.print_trainable_parameters()}")

## ══════════════════════════════════════════════════════════════
## TOKENIZE DATA

In [ ]:
def tokenize_chatml(example, max_length=512):
    """
    Tokenize ChatML format for Causal LM.
    Format: <|user|> message <|assistant|> response <|end|>
    """
    messages = example.get("messages", [])
    
    # Build text from messages
    text = ""
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        text += f"<|{role}|> {content}"
    text += "<|end|>"

In [ ]:
def tokenize_chatml(example, max_length=512):
    """
    Tokenize ChatML format for Causal LM training.
    """
    messages = example.get("messages", [])
    
    # Build text from messages
    text = ""
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        text += f"<|{role}|> {content} "
    text += "<|end|>"
    
    # Tokenize
    encoding = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    
    # Input IDs = everything except last token
    # Labels = everything except first token (causal LM shift)
    input_ids = encoding["input_ids"].squeeze()
    
    return {
        "input_ids": input_ids,
        "labels": input_ids.clone(),
        "attention_mask": encoding["attention_mask"].squeeze(),
    }

print("Tokenizing training data...")
train_tokenized = train_ds.map(
    tokenize_chatml,
    fn_kwargs={"max_length": config.max_seq_length},
    remove_columns=train_ds.column_names,
)

print("Tokenizing validation data...")
val_tokenized = val_ds.map(
    tokenize_chatml,
    fn_kwargs={"max_length": config.max_seq_length},
    remove_columns=val_ds.column_names,
)

print(f"Train tokens: {len(train_tokenized)}, Val tokens: {len(val_tokenized)}")

## ══════════════════════════════════════════════════════════════
## TRAINING — TRL SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# ─── Training arguments ───────────────────────────────────────
training_args = SFTConfig(
    output_dir=config.output_dir,
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.grad_accum_steps,
    warmup_steps=config.warmup_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    save_steps=config.save_steps,
    eval_steps=config.eval_steps,
    eval_strategy="steps",
    save_strategy="steps",
    fp16=False,
    bf16=True,  # Use BF16 on T4/A100
    optim="paged_adamw_8bit",  # Paged optimizer for memory efficiency
    max_grad_norm=0.3,
    report_to="wandb" if WANDB_KEY else "none",
    run_name=config.run_name,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    remove_unused_columns=False,
)

# ─── SFTTrainer ──────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    max_seq_length=config.max_seq_length,
    dataset_text_field="text",  # Required by SFTTrainer
)

# Show trainable params
trainer.model.print_trainable_parameters()

In [ ]:
# ─── TRAIN ───────────────────────────────────────────────────

print("Starting training...")
print(f"  Model: {config.model_name}")
print(f"  Epochs: {config.epochs}")
print(f"  Effective batch size: {config.batch_size * config.grad_accum_steps}")
print(f"  Max sequence length: {config.max_seq_length}")
print(f"  LoRA r: {config.lora_r}")
print(f"  Output dir: {config.output_dir}")
print(f"  Logging: {'wandb' if WANDB_KEY else 'disabled'}")

trainer.train()

print("\nTraining complete!")

In [ ]:
# ─── SAVE — merge LoRA and save full model ─────────────────────

import os

# Save adapter weights
trainer.save_model(config.output_dir)
print(f"Adapter saved to: {config.output_dir}")

# Merge and save full model (optional — for inference)
from peft import PeftModel

print("\nMerging LoRA adapters with base model...")
merged_output = f"{config.output_dir}-merged"

# Reload base model in FP16 for merging
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.bfloat16,
    device_map="cpu",  # Merge on CPU to avoid OOM
    trust_remote_code=True,
)

# Merge adapters
merged_model = PeftModel.from_pretrained(base_model, config.output_dir)
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(merged_output)
tokenizer.save_pretrained(merged_output)

print(f"\nMerged model saved to: {merged_output}")
print(f"  Model size: {os.path.getsize(merged_output)/1e9:.1f} GB")

## ══════════════════════════════════════════════════════════════
## EVALUATE — compute perplexity on validation set

In [ ]:
import math

# Run evaluation
print("Running evaluation...")
eval_results = trainer.evaluate()

print(f"\nEval Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

# Compute perplexity
if "eval_loss" in eval_results:
    perplexity = math.exp(eval_results["eval_loss"])
    print(f"  Perplexity: {perplexity:.2f}")

# ── wandb summary ─────────────────────────────────────────────
if WANDB_KEY:
    wandb.log({
        "eval_loss": eval_results.get("eval_loss", 0),
        "perplexity": perplexity if "eval_loss" in eval_results else None,
        "train_steps": trainer.state.global_step,
    })
    wandb.finish()

## ══════════════════════════════════════════════════════════════
## INFERENCE TEST — verify model generates coherent text

In [ ]:
# ─── Load merged model for inference ─────────────────────────

from transformers import pipeline

print(f"Loading merged model from: {merged_output}")
pipe = pipeline(
    "text-generation",
    model=merged_output,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# Test prompts
test_prompts = [
    "<|user|> Hello, how are you? <|end|>",
    "<|user|> Tell me about starfire. <|end|>",
]

print("\nGeneration tests:")
for prompt in test_prompts:
    print(f"\n[PROMPT] {prompt}")
    result = pipe(prompt)
    print(f"[OUTPUT] {result[0]['generated_text']}")

## ══════════════════════════════════════════════════════════════
## PUSH TO HUGGINGFACE (optional)

In [ ]:
# Push merged model to HuggingFace Hub
from huggingface_hub import HfApi

# Uncomment to push:
# api = HfApi()
# api.upload_folder(
#     folder_path=merged_output,
#     repo_id="your-username/qwen-0.8b-starfire",
#     repo_type="model",
#     token=HF_TOKEN,
# )
# print("Pushed to HuggingFace: your-username/qwen-0.8b-starfire")

print("\nTo push to HuggingFace, uncomment the push code above.")
print(f"Merged model at: {merged_output}")

## ══════════════════════════════════════════════════════════════
## NOTES

**Memory usage on T4:**
- QLoRA 4bit: ~10-12GB VRAM
- Effective batch size 16 (2×8 gradient accum)
- Fits comfortably on free T4

**Training time:**
- ~15-20 min per epoch on T4
- 3 epochs: ~45-60 min total

**LoRA rank tradeoffs:**
- r=8: faster training, less capacity
- r=16: good balance (used here)
- r=32: more capacity, slower, more memory

**wandb logging:**
- Set WANDB_API_KEY in Colab Secrets
- Dashboard: https://wandb.ai
- Tracks: loss curves, learning rate, GPU usage

**To use with different dataset:**
1. Override `config.dataset_path` with your dataset ID or local path
2. Adjust `format_starfire_conversation()` to match your data format
3. Adjust `tokenize_chatml()` if your format differs